In [ ]:
# CELL 1 - mount Drive (click the link, allow, paste code if asked)
from google.colab import drive
drive.mount('/content/drive')
print("mounted")


In [ ]:
# ============================================================
#  WHAT CHANGED IN DRIVE SINCE 21 AUG
# ============================================================
import os, json, datetime

ROOT   = "/content/drive/MyDrive"
CUTOFF = datetime.datetime(2026, 8, 20)      # <- anything newer than this is "new"
SKIP   = {".git", "__pycache__", ".ipynb_checkpoints", "qwen_cache", ".cache"}

def ts(p):
    try:    return datetime.datetime.fromtimestamp(os.path.getmtime(p))
    except OSError: return datetime.datetime(1970, 1, 1)

def mb(p):
    try:    return os.path.getsize(p) / 1e6
    except OSError: return 0.0

def is_new(p):
    return ts(p) > CUTOFF

# ---------------------------------------------------------------- PART 0
print("#" * 78)
print("# PART 0 - EVERY TOP-LEVEL ITEM IN MyDrive")
print("#" * 78)
try:
    top = sorted(os.listdir(ROOT))
except Exception as e:
    print("CANNOT READ DRIVE:", e); top = []

for name in top:
    p = os.path.join(ROOT, name)
    flag = " <-- NEW" if is_new(p) else ""
    if os.path.isdir(p):
        print(f"  DIR   {name:<44s} {ts(p):%Y-%m-%d %H:%M}{flag}")
    else:
        print(f"  file  {name:<44s} {ts(p):%Y-%m-%d %H:%M}  {mb(p):7.2f} MB{flag}")

# shared-with-me / shared drives live outside MyDrive
print("\n  --- other mount points under /content/drive ---")
DRIVE_ROOT = os.path.dirname(ROOT)
try:
    for name in sorted(os.listdir(DRIVE_ROOT)):
        if name == os.path.basename(ROOT):
            continue
        p2 = os.path.join(DRIVE_ROOT, name)
        print(f"  {name}")
        if os.path.isdir(p2):
            for sub in sorted(os.listdir(p2))[:25]:
                print(f"      {sub}")
except Exception as e:
    print("  (none / not readable:", e, ")")

# pick folders worth walking
KEYS = ("phase", "quran", "rag", "guardrail", "index", "output", "result", "model", "finetune")
CODE = (".py", ".ipynb", ".json", ".csv", ".bin", ".safetensors", ".pkl", ".npy")

def has_code(base, max_depth=2):
    for dirpath, dirnames, filenames in os.walk(base):
        dirnames[:] = [d for d in dirnames if d not in SKIP]
        if dirpath[len(base):].count(os.sep) >= max_depth:
            dirnames[:] = []
        if any(f.endswith(CODE) for f in filenames):
            return True
    return False

TARGETS = []
for n in top:
    p = os.path.join(ROOT, n)
    if not os.path.isdir(p) or n in SKIP:
        continue
    named = any(k in n.lower() for k in KEYS)
    if named or (is_new(p) and has_code(p)):
        TARGETS.append(n)

print(f"\n  -> walking {len(TARGETS)} folder(s): {TARGETS}")

# ---------------------------------------------------------------- PART 1
print()
print("#" * 78)
print("# PART 1 - FULL TREE  ( * = created/modified after 20 Aug )")
print("#" * 78)
for t in TARGETS:
    base = os.path.join(ROOT, t)
    print(f"\n=== {t} ===")
    for dirpath, dirnames, filenames in os.walk(base):
        dirnames[:] = [d for d in dirnames if d not in SKIP]
        depth = dirpath[len(base):].count(os.sep)
        if depth > 3:
            dirnames[:] = []
            continue
        pad = "  " * depth
        print(f"{pad}{os.path.basename(dirpath) or t}/")
        for fn in sorted(filenames)[:50]:
            fp = os.path.join(dirpath, fn)
            star = "*" if is_new(fp) else " "
            print(f"{pad} {star} {fn:<48s} {mb(fp):8.2f} MB  {ts(fp):%Y-%m-%d %H:%M}")
        if len(filenames) > 50:
            print(f"{pad}   ... +{len(filenames)-50} more files")

# ---------------------------------------------------------------- PART 2
print()
print("#" * 78)
print("# PART 2 - EVERYTHING NEW SINCE 20 AUG, NEWEST FIRST")
print("#" * 78)
fresh = []
for t in TARGETS:
    for dirpath, dirnames, filenames in os.walk(os.path.join(ROOT, t)):
        dirnames[:] = [d for d in dirnames if d not in SKIP]
        for fn in filenames:
            fp = os.path.join(dirpath, fn)
            if is_new(fp):
                fresh.append((ts(fp), mb(fp), fp.replace(ROOT + "/", "")))
fresh.sort(reverse=True)
if not fresh:
    print("\n  (nothing new - the re-run did not write into these folders)")
for t_, m_, p_ in fresh[:150]:
    print(f"  {t_:%Y-%m-%d %H:%M}  {m_:8.2f} MB  {p_}")
if len(fresh) > 150:
    print(f"  ... +{len(fresh)-150} more")

# ---------------------------------------------------------------- PART 3
print()
print("#" * 78)
print("# PART 3 - CONTENTS OF THE NEW RESULT FILES")
print("#" * 78)
shown = 0
for t_, m_, rel in fresh:
    if not rel.endswith((".json", ".csv", ".txt", ".md")):
        continue
    if m_ > 8:
        continue
    if shown >= 12:
        print("\n  (stopping at 12 files)"); break
    fp = os.path.join(ROOT, rel)
    print(f"\n--- {rel}  ({m_:.2f} MB) ---")
    shown += 1
    try:
        if rel.endswith(".json"):
            with open(fp, encoding="utf-8", errors="replace") as f:
                obj = json.load(f)
            if isinstance(obj, dict):
                print(f"    dict, {len(obj)} keys")
                txt = json.dumps(obj, ensure_ascii=False, indent=2)
                print("\n".join("    " + l for l in txt.splitlines()[:60]))
                if len(txt.splitlines()) > 60:
                    print("    ...")
            elif isinstance(obj, list):
                print(f"    list, {len(obj)} items; first item:")
                txt = json.dumps(obj[0] if obj else None, ensure_ascii=False, indent=2)
                print("\n".join("    " + l for l in txt.splitlines()[:35]))
        else:
            with open(fp, encoding="utf-8", errors="replace") as f:
                for i, line in enumerate(f):
                    if i >= 25: print("    ..."); break
                    print("    " + line.rstrip()[:160])
    except Exception as e:
        print("    could not read:", e)

# ---------------------------------------------------------------- PART 4
print()
print("#" * 78)
print("# PART 4 - WHICH LLM DID THE RE-RUN USE?")
print("#" * 78)
RETIRED = "llama-3.3-70b-versatile"
hits_old, hits_new = [], []
for t in TARGETS:
    for dirpath, dirnames, filenames in os.walk(os.path.join(ROOT, t)):
        dirnames[:] = [d for d in dirnames if d not in SKIP]
        for fn in filenames:
            if not fn.endswith((".py", ".ipynb", ".json", ".txt")):
                continue
            fp = os.path.join(dirpath, fn)
            if mb(fp) > 6:
                continue
            try:
                body = open(fp, encoding="utf-8", errors="replace").read()
            except Exception:
                continue
            rel = fp.replace(ROOT + "/", "")
            if RETIRED in body:
                hits_old.append(rel)
            for tag in ("allam-2-7b", "gpt-oss", "qwen3", "kimi", "moonshot",
                        "deepseek", "gemini", "gpt-4", "claude-"):
                if tag in body:
                    hits_new.append((rel, tag)); break

print(f"\n  still reference the RETIRED {RETIRED}:  {len(hits_old)}")
for r in hits_old[:20]:
    print("     ", r)
print(f"\n  reference some OTHER model:  {len(hits_new)}")
for r, tag in hits_new[:20]:
    print(f"      {tag:<12s} {r}")

# ---------------------------------------------------------------- PART 5
print()
print("#" * 78)
print("# PART 5 - INDEXES AND TRAINED MODELS PRESENT")
print("#" * 78)
for t in TARGETS:
    for dirpath, dirnames, filenames in os.walk(os.path.join(ROOT, t)):
        dirnames[:] = [d for d in dirnames if d not in SKIP]
        low = os.path.basename(dirpath).lower()
        marks = [f for f in filenames
                 if f in ("entries.json", "hnsw_index.bin", "config.json",
                          "model.safetensors", "pytorch_model.bin",
                          "modules.json", "index.bin")]
        if marks:
            star = "*" if is_new(dirpath) else " "
            print(f"  {star} {dirpath.replace(ROOT+'/','')}")
            print(f"      {sorted(marks)}   modified {ts(dirpath):%Y-%m-%d %H:%M}")

print("\n" + "#" * 78)
print("# DONE - copy EVERYTHING above and paste it back")
print("#" * 78)
